# morphological_quantification_2026-01-02 — 05_domain_quantification

**Feeds:** Fig 3f

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 05 | Domain Quantification

This notebook turns the whole-morph geometry from `02`, the posterior orientation from `03`, and the marker-positive masks from `04` into reviewer-facing domain measurements.


## Cell Guide

- `Setup`: resolve the project root, import helper code, and define output paths.
- `Settings Notes`: document how per-plane measurements are summarized across z.
- `Load Inputs`: read the consensus geometry, posterior clicks, and marker-positive masks from earlier notebooks.
- `Reconstruct Posterior-Oriented Consensus Axes`: rebuild the curved consensus axis for each file and orient it so the axis starts at posterior.
- `Quantify Marker Domains On Each Z Plane`: measure domain size, axis position, and bilaterality for each marker on each z plane.
- `Representative Domain Overlay Pages`: render a small number of image overlays with the posterior-oriented axis and marker domains.
- `Summarize Marker Domains Across All Z`: reduce per-plane measurements to one per-file summary using all z planes for each file.
- `Measurement Illustration Panels`: show one representative image plus a simple diagram before each family of summary plots.
- `Review Cohort-Level Domain Summaries`: inspect primary figures first, then supplementary descriptive figures.
- `Save Stage Outputs`: write the per-plane and per-file quantification tables and QC figures.
- `Next Step`: move to the final stats / figure-export notebook once these domain measurements look sensible.


In [ ]:
import json
import math
import sys
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().resolve().parent
elif (Path.cwd() / "notebooks").exists():
    ROOT = Path.cwd().resolve()
else:
    raise RuntimeError("Run this notebook from the project root or the notebooks/ directory.")

SCRIPTS_DIR = ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import morphology_quantification_helpers as mqh

pd.set_option("display.max_columns", 200)
plt.rcParams["figure.dpi"] = 120


## Settings Notes

- Domain measurements are computed on every available z plane from `04`.
- The fluorescent-domain summaries in this notebook also use every available z plane from `04`; the `02` z omissions are only used to stabilize construction of the single consensus axis per file.
- The curved consensus axis is reconstructed from the retained masks and then oriented so its first endpoint is the posterior end defined in `03`.
- Bilaterality is quantified relative to the posterior-oriented curved axis, using the sign of the local transverse offset from that axis.
- This notebook is intentionally focused on geometry and spatial organization. It is not the final statistics notebook.


In [ ]:
ANALYSIS_MANIFEST_PATH = ROOT / "results" / "manifests" / "analysis_manifest.tsv"
RAW_MANIFEST_PATH = ROOT / "results" / "manifests" / "raw_czi_manifest.tsv"
CONSENSUS_TABLE_PATH = ROOT / "results" / "tables" / "whole_morph_consensus_geometry.tsv"
PER_Z_GEOMETRY_TABLE_PATH = ROOT / "results" / "tables" / "whole_morph_per_z_geometry.tsv"
POSTERIOR_PATH = ROOT / "results" / "annotations" / "manual_posterior_clicks.tsv"
MARKER_PLANE_TABLE_PATH = ROOT / "results" / "tables" / "04_marker_positive_metrics_by_plane.tsv"

AXIS_TABLE_PATH = ROOT / "results" / "tables" / "05_posterior_oriented_consensus_axes.tsv"
DOMAIN_PLANE_TABLE_PATH = ROOT / "results" / "tables" / "05_marker_domain_metrics_by_plane.tsv"
DOMAIN_FILE_TABLE_PATH = ROOT / "results" / "tables" / "05_marker_domain_summary_by_file.tsv"
PAIRWISE_PLANE_TABLE_PATH = ROOT / "results" / "tables" / "05_marker_pairwise_metrics_by_plane.tsv"
PAIRWISE_FILE_TABLE_PATH = ROOT / "results" / "tables" / "05_marker_pairwise_summary_by_file.tsv"
LPM_SUMMARY_TABLE_PATH = ROOT / "results" / "tables" / "05c_lpm_size_summary_by_file.tsv"
MESP2_SUMMARY_TABLE_PATH = ROOT / "results" / "tables" / "05d_mesp2_expression_summary_by_file.tsv"
PAX8_SUMMARY_TABLE_PATH = ROOT / "results" / "tables" / "05e_pax8_expression_summary_by_file.tsv"

DOMAIN_QC_DIR = ROOT / "results" / "qc" / "domain_quantification_review"
DOMAIN_QC_DIR.mkdir(parents=True, exist_ok=True)
DOMAIN_OVERLAY_PATH = DOMAIN_QC_DIR / "05_representative_domain_overlays.png"
DOMAIN_SIZE_EXPLAINER_PATH = DOMAIN_QC_DIR / "05_domain_size_measurement_explainer.png"
DOMAIN_SIZE_SUMMARY_PATH = DOMAIN_QC_DIR / "05_marker_domain_size_summary.png"
DOMAIN_SIZE_SUPPLEMENT_PATH = DOMAIN_QC_DIR / "05_marker_domain_size_supplement.png"
DOMAIN_POSITION_EXPLAINER_PATH = DOMAIN_QC_DIR / "05_axis_position_measurement_explainer.png"
DOMAIN_POSITION_SUMMARY_PATH = DOMAIN_QC_DIR / "05_marker_domain_axis_position_summary.png"
DOMAIN_POSITION_SUPPLEMENT_PATH = DOMAIN_QC_DIR / "05_marker_domain_axis_position_supplement.png"
DOMAIN_BILATERALITY_EXPLAINER_PATH = DOMAIN_QC_DIR / "05_bilaterality_measurement_explainer.png"
DOMAIN_BILATERALITY_SUMMARY_PATH = DOMAIN_QC_DIR / "05_marker_domain_bilaterality_summary.png"
DOMAIN_BILATERALITY_SUPPLEMENT_PATH = DOMAIN_QC_DIR / "05_marker_domain_bilaterality_supplement.png"
PAIRWISE_EXPLAINER_PATH = DOMAIN_QC_DIR / "05_pairwise_measurement_explainer.png"
PAIRWISE_SUMMARY_PATH = DOMAIN_QC_DIR / "05_pairwise_domain_summary.png"

PIXEL_SIZE_FALLBACK_UM = 1.3
CENTERLINE_PROJECTION_SAMPLES = 241
INLINE_REPRESENTATIVE_DOMAIN_FIGURES = 1
REPRESENTATIVE_FILE_COUNT = 3
OPTIMIZATION_MODE = False


## Load Inputs


In [ ]:
manifest_df = pd.read_csv(ANALYSIS_MANIFEST_PATH, sep="\t")
included_file_paths = set(
    manifest_df.loc[
        manifest_df["include_in_analysis"].fillna(True).astype(bool),
        "file_path",
    ].astype(str)
)

raw_manifest_df = pd.read_csv(RAW_MANIFEST_PATH, sep="\t")
raw_manifest_df = raw_manifest_df.loc[
    raw_manifest_df["file_path"].astype(str).isin(included_file_paths)
].copy()

consensus_df = pd.read_csv(CONSENSUS_TABLE_PATH, sep="\t")
consensus_df = consensus_df.loc[
    consensus_df["file_path"].astype(str).isin(included_file_paths)
].copy()

per_z_geometry_df = pd.read_csv(PER_Z_GEOMETRY_TABLE_PATH, sep="\t")
per_z_geometry_df = per_z_geometry_df.loc[
    per_z_geometry_df["file_path"].astype(str).isin(included_file_paths)
].copy()

posterior_df = pd.read_csv(POSTERIOR_PATH, sep="\t")
posterior_df = posterior_df.loc[
    posterior_df["file_path"].astype(str).isin(included_file_paths)
].copy()

marker_plane_df = pd.read_csv(MARKER_PLANE_TABLE_PATH, sep="\t")
marker_plane_df = marker_plane_df.loc[
    marker_plane_df["file_path"].astype(str).isin(included_file_paths)
].copy()

axis_input_df = mqh.build_posterior_annotation_input_table(consensus_df, per_z_geometry_df).merge(
    posterior_df[["file_path", "posterior_click_x_px", "posterior_click_y_px"]],
    on="file_path",
    how="inner",
).merge(
    raw_manifest_df[["file_path", "pixel_size_x_um"]].drop_duplicates("file_path"),
    on="file_path",
    how="left",
)

if len(axis_input_df) != len(consensus_df):
    missing = sorted(set(consensus_df["file_path"].astype(str)) - set(axis_input_df["file_path"].astype(str)))
    raise RuntimeError(
        "Missing posterior / axis input rows for file paths: " + ", ".join(missing[:5])
    )

print(f"Included files: {len(included_file_paths)}")
print(f"Consensus rows: {len(consensus_df)}")
print(f"Posterior clicks: {len(posterior_df)}")
print(f"Marker-positive plane rows: {len(marker_plane_df)}")


## Reconstruct Posterior-Oriented Consensus Axes

The summary table from `02` stores the axis endpoints and lengths, but the actual domain measurements below need the full curved axis. So here we rebuild the consensus centerline from the retained masks for each file, then orient it so the first endpoint is posterior according to the manual click from `03`.


In [ ]:
axis_records = []
axis_cache: dict[str, dict] = {}
retained_z_map: dict[str, set[int]] = {}
file_pixel_size_um: dict[str, float] = {}


for row in axis_input_df.itertuples(index=False):
    pixel_size_um = float(getattr(row, "pixel_size_x_um", np.nan))
    if not np.isfinite(pixel_size_um):
        pixel_size_um = float(PIXEL_SIZE_FALLBACK_UM)
    posterior_click_xy = np.array(
        [float(row.posterior_click_x_px), float(row.posterior_click_y_px)],
        dtype=np.float64,
    )

    consensus_geometry = mqh.consensus_geometry_from_annotation_row(
        row=row,
        per_z_geometry_df=per_z_geometry_df,
        root=ROOT,
        average_mode="mean",
        n_points=CENTERLINE_PROJECTION_SAMPLES,
    )
    oriented_centerline_xy = mqh.orient_centerline_to_reference_end(
        consensus_geometry["centerline_xy"],
        posterior_click_xy,
    )
    oriented_geometry = mqh.centerline_dict_from_polyline(
        oriented_centerline_xy,
        mask=None,
        method=str(consensus_geometry["method"]) + "_posterior_oriented",
    )

    file_path = str(row.file_path)
    retained_z = set(mqh.parse_int_list_field(getattr(row, "retained_z_indices", "")))
    if not retained_z:
        retained_z = {int(row.display_z_index)}

    retained_z_map[file_path] = retained_z
    file_pixel_size_um[file_path] = pixel_size_um
    axis_cache[file_path] = {
        **oriented_geometry,
        "posterior_click_xy": posterior_click_xy,
        "pixel_size_um": pixel_size_um,
        "display_z_index": int(row.display_z_index),
        "file_id": int(row.file_id),
        "retained_z_indices": sorted(int(v) for v in retained_z),
    }

    axis_records.append(
        {
            "image_id": str(row.image_id),
            "cohort_id": str(row.cohort_id),
            "canonical_position": int(row.canonical_position),
            "file_id": int(row.file_id),
            "file_path": file_path,
            "display_z_index": int(row.display_z_index),
            "retained_z_indices": ",".join(str(int(v)) for v in sorted(retained_z)),
            "posterior_click_x_px": float(posterior_click_xy[0]),
            "posterior_click_y_px": float(posterior_click_xy[1]),
            "pixel_size_um": float(pixel_size_um),
            "axis_method": str(oriented_geometry["method"]),
            "axis_length_px": float(oriented_geometry["length_px"]),
            "axis_length_um": float(oriented_geometry["length_px"] * pixel_size_um),
            "axis_tortuosity": float(oriented_geometry["tortuosity"]),
            "posterior_endpoint_x_px": float(oriented_geometry["endpoint_a_xy"][0]),
            "posterior_endpoint_y_px": float(oriented_geometry["endpoint_a_xy"][1]),
            "anterior_endpoint_x_px": float(oriented_geometry["endpoint_b_xy"][0]),
            "anterior_endpoint_y_px": float(oriented_geometry["endpoint_b_xy"][1]),
            "axis_midpoint_x_px": float(oriented_geometry["midpoint_xy"][0]),
            "axis_midpoint_y_px": float(oriented_geometry["midpoint_xy"][1]),
            "centerline_xy_json": json.dumps(oriented_geometry["centerline_xy"].tolist()),
        }
    )

axis_df = pd.DataFrame(axis_records).sort_values(["file_id"]).reset_index(drop=True)
axis_df.to_csv(AXIS_TABLE_PATH, sep="\t", index=False)
display(axis_df[["file_id", "axis_length_um", "axis_tortuosity", "retained_z_indices"]].style.hide(axis="index"))
print("Wrote posterior-oriented axis table:", AXIS_TABLE_PATH.relative_to(ROOT).as_posix())


## Quantify Marker Domains On Each Z Plane

For every marker-positive mask from `04`, we project the positive pixels onto the posterior-oriented curved axis and compute:

- domain size
- posterior/anterior extent along the axis
- centroid position along the axis
- simple bilaterality relative to the curved axis

We also compute a small set of pairwise marker relationships from those per-plane domain intervals.


In [ ]:
@lru_cache(maxsize=4096)
def load_binary_mask_cached(rel_path: str) -> np.ndarray:
    return mqh.load_binary_mask(ROOT / str(rel_path))


domain_rows = []
pairwise_rows = []
pair_specs = [
    ("foxf1", "pax8"),
    ("pax8", "mesp2"),
    ("foxf1", "mesp2"),
]
current_file_path = None
current_stack = None
current_channel_idx_map = None


for plane_row in marker_plane_df.sort_values(["file_id", "z_index", "marker_key"]).itertuples(index=False):
    file_path = str(plane_row.file_path)
    axis_info = axis_cache.get(file_path)
    if axis_info is None:
        continue
    if file_path != current_file_path:
        current_stack = mqh.load_czi_stack(ROOT / str(file_path))
        current_channel_idx_map = mqh.marker_channel_index_map(current_stack.channels)
        current_file_path = file_path

    positive_mask = load_binary_mask_cached(str(plane_row.positive_mask_path))
    organoid_mask = load_binary_mask_cached(str(plane_row.mask_path))
    pixel_size_um = float(axis_info["pixel_size_um"])
    z_index = int(plane_row.z_index)
    threshold_value = float(plane_row.threshold_value)
    retained_z = retained_z_map.get(file_path, set())
    is_retained_z = bool(z_index in retained_z)

    ys, xs = np.where(positive_mask)
    positive_points_xy = np.column_stack([xs, ys]).astype(np.float64) if len(xs) else np.zeros((0, 2), dtype=np.float64)
    if positive_points_xy.shape[0] > 0:
        marker_idx = None if current_channel_idx_map is None else current_channel_idx_map.get(str(plane_row.marker_key))
        if marker_idx is None or current_stack is None:
            raise RuntimeError(
                f"Missing marker plane for marker={plane_row.marker_key} file={file_path} z={z_index}"
            )
        marker_plane = np.asarray(current_stack.data_czyx[int(marker_idx), z_index], dtype=np.float32)
        corrected_signal, _, _ = mqh.background_correct_signal(
            signal=marker_plane,
            organoid_mask=organoid_mask,
            background_estimator=str(plane_row.background_estimator),
        )
        positive_weights = np.maximum(
            corrected_signal[ys, xs].astype(np.float64) - float(threshold_value),
            0.0,
        )
        projection = mqh.project_points_to_centerline(
            positive_points_xy,
            axis_info["centerline_xy"],
            n_samples=CENTERLINE_PROJECTION_SAMPLES,
        )
        arc_px = np.asarray(projection["projected_arc_px"], dtype=np.float64)
        arc_fraction = np.asarray(projection["projected_fraction"], dtype=np.float64)
        signed_px = np.asarray(projection["signed_transverse_px"], dtype=np.float64)
        abs_px = np.asarray(projection["transverse_abs_px"], dtype=np.float64)

        left_count = int(np.sum(signed_px < -0.5))
        right_count = int(np.sum(signed_px > 0.5))
        center_count = int(positive_points_xy.shape[0] - left_count - right_count)
        side_den = left_count + right_count
        side_balance = (
            1.0 - abs(float(right_count - left_count)) / float(side_den)
            if side_den > 0
            else np.nan
        )
        side_dominance = (
            float(right_count - left_count) / float(side_den)
            if side_den > 0
            else np.nan
        )
        left_weight_sum = float(np.sum(positive_weights[signed_px < -0.5]))
        right_weight_sum = float(np.sum(positive_weights[signed_px > 0.5]))
        center_weight_sum = float(
            np.sum(positive_weights[(signed_px >= -0.5) & (signed_px <= 0.5)])
        )
        weighted_side_den = float(left_weight_sum + right_weight_sum)
        weighted_side_balance = (
            1.0 - abs(float(right_weight_sum - left_weight_sum)) / float(weighted_side_den)
            if weighted_side_den > 0
            else np.nan
        )

        axis_min_px = float(np.nanmin(arc_px))
        axis_p10_px = float(np.nanpercentile(arc_px, 10.0))
        axis_max_px = float(np.nanmax(arc_px))
        axis_min_fraction = float(np.nanmin(arc_fraction))
        axis_p10_fraction = float(np.nanpercentile(arc_fraction, 10.0))
        axis_max_fraction = float(np.nanmax(arc_fraction))
        axis_centroid_px = float(np.nanmean(arc_px))
        axis_centroid_fraction = float(np.nanmean(arc_fraction))
        axis_span_px = float(axis_max_px - axis_min_px)
        axis_span_fraction = float(axis_max_fraction - axis_min_fraction)
        centroid_x_px = float(np.nanmean(positive_points_xy[:, 0]))
        centroid_y_px = float(np.nanmean(positive_points_xy[:, 1]))
        mean_abs_transverse_px = float(np.nanmean(abs_px))
    else:
        left_count = right_count = center_count = 0
        side_balance = np.nan
        side_dominance = np.nan
        left_weight_sum = right_weight_sum = center_weight_sum = 0.0
        weighted_side_balance = np.nan
        axis_min_px = np.nan
        axis_p10_px = np.nan
        axis_max_px = np.nan
        axis_min_fraction = np.nan
        axis_p10_fraction = np.nan
        axis_max_fraction = np.nan
        axis_centroid_px = np.nan
        axis_centroid_fraction = np.nan
        axis_span_px = np.nan
        axis_span_fraction = np.nan
        centroid_x_px = np.nan
        centroid_y_px = np.nan
        mean_abs_transverse_px = np.nan

    axis_length_px = float(axis_info["length_px"])
    domain_rows.append(
        {
            "image_id": str(plane_row.image_id),
            "cohort_id": str(plane_row.cohort_id),
            "canonical_position": int(plane_row.canonical_position),
            "file_id": int(plane_row.file_id),
            "file_path": file_path,
            "acquisition_date": str(plane_row.acquisition_date),
            "acquisition_batch_label": str(plane_row.acquisition_batch_label),
            "z_index": z_index,
            "z_count": int(plane_row.z_count),
            "is_retained_z": bool(is_retained_z),
            "display_z_index": int(axis_info["display_z_index"]),
            "marker_key": str(plane_row.marker_key),
            "marker_display_name": str(plane_row.marker_display_name),
            "threshold_sigma_multiple": float(plane_row.threshold_sigma_multiple),
            "threshold_value": threshold_value,
            "background_estimator": str(plane_row.background_estimator),
            "organoid_pixel_count": int(plane_row.organoid_pixel_count),
            "positive_pixels": int(plane_row.positive_pixels),
            "positive_fraction": float(plane_row.positive_fraction),
            "positive_area_um2": float(plane_row.positive_pixels) * (pixel_size_um**2),
            "axis_length_px": axis_length_px,
            "axis_length_um": axis_length_px * pixel_size_um,
            "posterior_extent_px": axis_min_px,
            "posterior_extent_um": axis_min_px * pixel_size_um if np.isfinite(axis_min_px) else np.nan,
            "posterior_extent_fraction": axis_min_fraction,
            "axis_p10_px": axis_p10_px,
            "axis_p10_um": axis_p10_px * pixel_size_um if np.isfinite(axis_p10_px) else np.nan,
            "axis_p10_fraction": axis_p10_fraction,
            "anterior_extent_px": axis_max_px,
            "anterior_extent_um": axis_max_px * pixel_size_um if np.isfinite(axis_max_px) else np.nan,
            "anterior_extent_fraction": axis_max_fraction,
            "axis_span_px": axis_span_px,
            "axis_span_um": axis_span_px * pixel_size_um if np.isfinite(axis_span_px) else np.nan,
            "axis_span_fraction": axis_span_fraction,
            "axis_centroid_px": axis_centroid_px,
            "axis_centroid_um": axis_centroid_px * pixel_size_um if np.isfinite(axis_centroid_px) else np.nan,
            "axis_centroid_fraction": axis_centroid_fraction,
            "centroid_x_px": centroid_x_px,
            "centroid_y_px": centroid_y_px,
            "mean_abs_transverse_px": mean_abs_transverse_px,
            "mean_abs_transverse_um": mean_abs_transverse_px * pixel_size_um if np.isfinite(mean_abs_transverse_px) else np.nan,
            "left_positive_pixels": left_count,
            "right_positive_pixels": right_count,
            "centerline_positive_pixels": center_count,
            "left_positive_weighted_signal": left_weight_sum,
            "right_positive_weighted_signal": right_weight_sum,
            "centerline_positive_weighted_signal": center_weight_sum,
            "side_balance_index": side_balance,
            "weighted_side_balance_index": weighted_side_balance,
            "side_dominance_score": side_dominance,
            "positive_mask_path": str(plane_row.positive_mask_path),
            "mask_path": str(plane_row.mask_path),
        }
    )

domain_plane_df = pd.DataFrame(domain_rows).sort_values(
    ["file_id", "z_index", "marker_key"]
).reset_index(drop=True)
for prefix in ["posterior_extent", "axis_p10", "anterior_extent", "axis_span", "axis_centroid"]:
    frac_col = f"{prefix}_fraction"
    if frac_col in domain_plane_df.columns:
        domain_plane_df[f"{prefix}_um"] = (
            domain_plane_df[frac_col].astype(float) * domain_plane_df["axis_length_um"].astype(float)
        )
domain_plane_df.to_csv(DOMAIN_PLANE_TABLE_PATH, sep="\t", index=False)

for (file_id, z_index), plane_sub in domain_plane_df.groupby(["file_id", "z_index"], sort=True):
    lookup = {str(row.marker_key): row for row in plane_sub.itertuples(index=False)}
    for marker_a, marker_b in pair_specs:
        row_a = lookup.get(marker_a)
        row_b = lookup.get(marker_b)
        if row_a is None or row_b is None:
            continue
        overlap_fraction = np.nan
        if np.isfinite(getattr(row_a, "axis_span_fraction")) and np.isfinite(getattr(row_b, "axis_span_fraction")):
            overlap_start = max(float(row_a.posterior_extent_fraction), float(row_b.posterior_extent_fraction))
            overlap_end = min(float(row_a.anterior_extent_fraction), float(row_b.anterior_extent_fraction))
            overlap_fraction = max(0.0, overlap_end - overlap_start)
        pairwise_rows.append(
            {
                "file_id": int(file_id),
                "file_path": str(row_a.file_path),
                "z_index": int(z_index),
                "is_retained_z": bool(row_a.is_retained_z and row_b.is_retained_z),
                "marker_a_key": marker_a,
                "marker_b_key": marker_b,
                "marker_a_display_name": str(row_a.marker_display_name),
                "marker_b_display_name": str(row_b.marker_display_name),
                "marker_a_positive_fraction": float(row_a.positive_fraction),
                "marker_b_positive_fraction": float(row_b.positive_fraction),
                "centroid_delta_fraction_b_minus_a": (
                    float(row_b.axis_centroid_fraction) - float(row_a.axis_centroid_fraction)
                    if np.isfinite(row_a.axis_centroid_fraction) and np.isfinite(row_b.axis_centroid_fraction)
                    else np.nan
                ),
                "posterior_extent_delta_fraction_b_minus_a": (
                    float(row_b.posterior_extent_fraction) - float(row_a.posterior_extent_fraction)
                    if np.isfinite(row_a.posterior_extent_fraction) and np.isfinite(row_b.posterior_extent_fraction)
                    else np.nan
                ),
                "anterior_extent_delta_fraction_b_minus_a": (
                    float(row_b.anterior_extent_fraction) - float(row_a.anterior_extent_fraction)
                    if np.isfinite(row_a.anterior_extent_fraction) and np.isfinite(row_b.anterior_extent_fraction)
                    else np.nan
                ),
                "axis_interval_overlap_fraction": overlap_fraction,
            }
        )

pairwise_plane_df = pd.DataFrame(pairwise_rows).sort_values(
    ["file_id", "z_index", "marker_a_key", "marker_b_key"]
).reset_index(drop=True)
pairwise_plane_df.to_csv(PAIRWISE_PLANE_TABLE_PATH, sep="\t", index=False)

display(
    domain_plane_df[
        [
            "file_id",
            "z_index",
            "is_retained_z",
            "marker_display_name",
            "positive_fraction",
            "axis_centroid_fraction",
            "axis_span_fraction",
            "side_balance_index",
            "weighted_side_balance_index",
        ]
    ].head(12).style.hide(axis="index")
)
print("Wrote domain metrics by plane:", DOMAIN_PLANE_TABLE_PATH.relative_to(ROOT).as_posix())
print("Wrote pairwise metrics by plane:", PAIRWISE_PLANE_TABLE_PATH.relative_to(ROOT).as_posix())


## Representative Domain Overlay Pages

These overlays show a small number of representative files with:

- the posterior-oriented consensus axis
- posterior and anterior endpoints
- marker-positive boundaries
- marker domain centroids


In [ ]:
@lru_cache(maxsize=256)
def load_plane_bundle(file_path: str, z_index: int) -> dict:
    abs_path = ROOT / str(file_path)
    return mqh.load_plane_channels(abs_path, z_index=int(z_index))


def representative_file_ids(file_ids: list[int], n_examples: int = 3) -> list[int]:
    if not file_ids:
        return []
    unique_ids = sorted(set(int(v) for v in file_ids))
    if len(unique_ids) <= n_examples:
        return unique_ids
    idx_values = np.linspace(0, len(unique_ids) - 1, n_examples)
    return [unique_ids[int(round(float(idx)))] for idx in idx_values]


representative_ids = representative_file_ids(domain_plane_df["file_id"].astype(int).tolist(), n_examples=REPRESENTATIVE_FILE_COUNT)
fig, axes = plt.subplots(
    len(representative_ids),
    1,
    figsize=(8.8, 4.6 * max(len(representative_ids), 1)),
    constrained_layout=True,
)
axes = np.atleast_1d(np.asarray(axes)).ravel()

for ax, file_id in zip(axes, representative_ids):
    file_sub = domain_plane_df.loc[
        (domain_plane_df["file_id"].astype(int) == int(file_id))
        & (domain_plane_df["is_retained_z"].astype(bool))
    ].copy()
    if file_sub.empty:
        ax.axis("off")
        continue
    rep_rows = file_sub.loc[file_sub["z_index"].astype(int) == int(file_sub["display_z_index"].iloc[0])]
    if rep_rows.empty:
        rep_rows = file_sub.loc[file_sub["z_index"].astype(int) == int(file_sub["z_index"].median())]
    rep_rows = rep_rows.sort_values("marker_key").reset_index(drop=True)
    first = rep_rows.iloc[0]
    plane = load_plane_bundle(str(first.file_path), int(first.z_index))
    mask = load_binary_mask_cached(str(first.mask_path))
    axis_info = axis_cache[str(first.file_path)]

    ax.imshow(mqh.robust_rescale(plane["dapi"]), cmap="gray", interpolation="nearest")
    mqh.plot_mask_outline(ax, mask, color="white", linewidth=0.9)
    mqh.plot_centerline_overlay(
        ax=ax,
        centerline_xy=axis_info["centerline_xy"],
        midpoint_xy=axis_info["midpoint_xy"],
        endpoint_a_xy=axis_info["endpoint_a_xy"],
        endpoint_b_xy=axis_info["endpoint_b_xy"],
        base_endpoint_a_xy=axis_info["base_endpoint_a_xy"],
        base_endpoint_b_xy=axis_info["base_endpoint_b_xy"],
        posterior_click_xy=np.asarray(axis_info["posterior_click_xy"], dtype=np.float64),
        line_width=1.2,
    )

    label_lines = []
    for row in rep_rows.itertuples(index=False):
        positive_mask = load_binary_mask_cached(str(row.positive_mask_path))
        mqh.plot_mask_outline(ax, positive_mask, color=mqh.MARKER_OUTLINE_COLORS[str(row.marker_key)], linewidth=1.2)
        if np.isfinite(row.centroid_x_px) and np.isfinite(row.centroid_y_px):
            ax.scatter(
                [float(row.centroid_x_px)],
                [float(row.centroid_y_px)],
                s=20,
                c=[mqh.MARKER_OUTLINE_COLORS[str(row.marker_key)]],
                edgecolors="white",
                linewidths=0.4,
            )
        label_lines.append(
            f"{str(row.marker_display_name)}: frac={float(row.positive_fraction):.2f}, axis={float(row.axis_centroid_fraction):.2f}"
        )

    ax.set_title(
        f"file {int(file_id):02d} | z={int(first.z_index):02d} | representative retained-z domain overlay",
        fontsize=10.5,
    )
    ax.set_xticks([])
    ax.set_yticks([])
    ax.text(
        1.01,
        0.5,
        "\n".join(label_lines),
        transform=ax.transAxes,
        ha="left",
        va="center",
        fontsize=8.3,
    )

for ax in axes[len(representative_ids):]:
    ax.axis("off")

fig.savefig(DOMAIN_OVERLAY_PATH, dpi=180, bbox_inches="tight")
if not OPTIMIZATION_MODE and INLINE_REPRESENTATIVE_DOMAIN_FIGURES > 0:
    display(fig)
plt.close(fig)
print("Wrote representative domain overlay figure:", DOMAIN_OVERLAY_PATH.relative_to(ROOT).as_posix())


## Summarize Marker Domains Across All Z

The final per-file summaries in this notebook use all available z planes for each file. For each marker, we keep the median all-z value as the main summary and also save the min/max all-z range for stability.


In [ ]:
all_z_domain_df = domain_plane_df.copy()

summary_metrics = [
    "positive_pixels",
    "positive_fraction",
    "positive_area_um2",
    "posterior_extent_fraction",
    "axis_p10_fraction",
    "anterior_extent_fraction",
    "axis_span_fraction",
    "axis_centroid_fraction",
    "mean_abs_transverse_um",
    "side_balance_index",
    "weighted_side_balance_index",
]

summary_rows = []
for (file_id, file_path, marker_key), sub in all_z_domain_df.groupby(
    ["file_id", "file_path", "marker_key"], sort=True
):
    sub = sub.sort_values("z_index").reset_index(drop=True)
    first = sub.iloc[0]
    out = {
        "file_id": int(file_id),
        "file_path": str(file_path),
        "canonical_position": int(first["canonical_position"]),
        "acquisition_date": str(first["acquisition_date"]),
        "acquisition_batch_label": str(first["acquisition_batch_label"]),
        "marker_key": str(marker_key),
        "marker_display_name": str(first["marker_display_name"]),
        "n_z_planes_used": int(len(sub)),
        "z_indices_used": ",".join(str(int(v)) for v in sub["z_index"].astype(int).tolist()),
        "display_z_index": int(first["display_z_index"]),
        "axis_length_um": float(sub["axis_length_um"].median()),
    }
    for metric in summary_metrics:
        values = sub[metric].astype(float).to_numpy()
        out[f"{metric}_median"] = float(np.nanmedian(values))
        out[f"{metric}_min"] = float(np.nanmin(values)) if np.isfinite(values).any() else np.nan
        out[f"{metric}_max"] = float(np.nanmax(values)) if np.isfinite(values).any() else np.nan
        out[f"{metric}_range"] = (
            float(np.nanmax(values) - np.nanmin(values))
            if np.isfinite(values).any()
            else np.nan
        )
    summary_rows.append(out)

domain_file_df = pd.DataFrame(summary_rows).sort_values(
    ["marker_key", "file_id"]
).reset_index(drop=True)
for prefix in ["posterior_extent", "axis_p10", "anterior_extent", "axis_span", "axis_centroid"]:
    for suffix in ["median", "min", "max", "range"]:
        frac_col = f"{prefix}_fraction_{suffix}"
        if frac_col in domain_file_df.columns:
            domain_file_df[f"{prefix}_um_{suffix}"] = (
                domain_file_df[frac_col].astype(float) * domain_file_df["axis_length_um"].astype(float)
            )
domain_file_df.to_csv(DOMAIN_FILE_TABLE_PATH, sep="\t", index=False)

all_z_pairwise_df = pairwise_plane_df.copy()
pairwise_summary_rows = []
pair_summary_metrics = [
    "centroid_delta_fraction_b_minus_a",
    "posterior_extent_delta_fraction_b_minus_a",
    "anterior_extent_delta_fraction_b_minus_a",
    "axis_interval_overlap_fraction",
]
for (file_id, file_path, marker_a_key, marker_b_key), sub in all_z_pairwise_df.groupby(
    ["file_id", "file_path", "marker_a_key", "marker_b_key"], sort=True
):
    sub = sub.sort_values("z_index").reset_index(drop=True)
    first = sub.iloc[0]
    out = {
        "file_id": int(file_id),
        "file_path": str(file_path),
        "marker_a_key": str(marker_a_key),
        "marker_b_key": str(marker_b_key),
        "marker_a_display_name": str(first["marker_a_display_name"]),
        "marker_b_display_name": str(first["marker_b_display_name"]),
        "n_z_planes_used": int(len(sub)),
        "z_indices_used": ",".join(str(int(v)) for v in sub["z_index"].astype(int).tolist()),
    }
    for metric in pair_summary_metrics:
        values = sub[metric].astype(float).to_numpy()
        out[f"{metric}_median"] = float(np.nanmedian(values))
        out[f"{metric}_min"] = float(np.nanmin(values)) if np.isfinite(values).any() else np.nan
        out[f"{metric}_max"] = float(np.nanmax(values)) if np.isfinite(values).any() else np.nan
    pairwise_summary_rows.append(out)

pairwise_file_df = pd.DataFrame(pairwise_summary_rows).sort_values(
    ["marker_a_key", "marker_b_key", "file_id"]
).reset_index(drop=True)
pairwise_file_df.to_csv(PAIRWISE_FILE_TABLE_PATH, sep="\t", index=False)

display(
    domain_file_df[
        [
            "file_id",
            "marker_display_name",
            "positive_fraction_median",
            "axis_centroid_fraction_median",
            "axis_span_fraction_median",
            "side_balance_index_median",
            "weighted_side_balance_index_median",
        ]
    ].head(12).style.hide(axis="index")
)
print("Wrote domain summary by file:", DOMAIN_FILE_TABLE_PATH.relative_to(ROOT).as_posix())
print("Wrote pairwise summary by file:", PAIRWISE_FILE_TABLE_PATH.relative_to(ROOT).as_posix())
print("Wrote LPM summary by file:", LPM_SUMMARY_TABLE_PATH.relative_to(ROOT).as_posix())
print("Wrote MESP2 summary by file:", MESP2_SUMMARY_TABLE_PATH.relative_to(ROOT).as_posix())
print("Wrote PAX8 summary by file:", PAX8_SUMMARY_TABLE_PATH.relative_to(ROOT).as_posix())


## Review Cohort-Level Domain Summaries

These quick summary plots are not the final manuscript statistics. The notebook now separates:

- primary readouts that map directly onto the original target list for this assay
- supplementary descriptive readouts that are still useful, but were not part of the original must-have list

Each plot family is preceded by one representative image plus a simple diagram showing what the plotted metric means, then followed by the cohort-level summary.


In [ ]:
marker_order = ["mesp2", "pax8", "foxf1"]
marker_labels = [mqh.marker_display_name(key) for key in marker_order]
x_positions = np.arange(len(marker_order), dtype=float)

def plot_marker_metric_panel(ax, metric_name: str, title: str):
    for idx, marker_key in enumerate(marker_order):
        sub = domain_file_df.loc[domain_file_df["marker_key"] == marker_key].copy()
        if sub.empty:
            continue
        y = sub[metric_name].astype(float).to_numpy()
        jitter = np.linspace(-0.12, 0.12, len(y)) if len(y) > 1 else np.array([0.0], dtype=float)
        ax.scatter(
            np.full(len(y), x_positions[idx]) + jitter,
            y,
            color=mqh.MARKER_OUTLINE_COLORS[marker_key],
            alpha=0.8,
            s=28,
            edgecolors="white",
            linewidths=0.3,
        )
        ax.hlines(float(np.nanmedian(y)), x_positions[idx] - 0.20, x_positions[idx] + 0.20, color="black", linewidth=2.0)
    ax.set_xticks(x_positions, marker_labels, rotation=15, ha="right")
    ax.set_title(title, fontsize=10.5)
    ax.grid(alpha=0.18, axis="y")

def draw_mask_fill(ax, mask: np.ndarray, color, alpha: float = 0.35):
    rgba = np.zeros((*mask.shape, 4), dtype=np.float32)
    rgba[..., :3] = plt.matplotlib.colors.to_rgb(color)
    rgba[..., 3] = np.where(np.asarray(mask, dtype=bool), float(alpha), 0.0)
    ax.imshow(rgba, interpolation="nearest")

def configure_geometry_panel(ax, mask: np.ndarray, pad: float = 18.0):
    h, w = np.asarray(mask).shape[:2]
    ax.set_xlim(-pad, w + pad)
    ax.set_ylim(h + pad, -pad)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

def centerline_point_from_fraction(centerline_xy: np.ndarray, fraction: float) -> np.ndarray:
    centerline_xy = np.asarray(centerline_xy, dtype=np.float64)
    if centerline_xy.ndim != 2 or centerline_xy.shape[0] < 2 or not np.isfinite(float(fraction)):
        return np.array([np.nan, np.nan], dtype=np.float64)
    arc = mqh._polyline_arc_length_xy(centerline_xy)
    total = float(arc[-1]) if arc.size else 0.0
    if total <= 0:
        return np.array([np.nan, np.nan], dtype=np.float64)
    return mqh._interpolate_point_along_polyline(
        centerline_xy,
        float(np.clip(fraction, 0.0, 1.0)) * total,
    )

def centerline_segment_xy(
    centerline_xy: np.ndarray,
    start_fraction: float,
    end_fraction: float,
    n_points: int = 301,
) -> np.ndarray:
    centerline_xy = np.asarray(centerline_xy, dtype=np.float64)
    if centerline_xy.ndim != 2 or centerline_xy.shape[0] < 2:
        return np.zeros((0, 2), dtype=np.float64)
    if not np.isfinite(float(start_fraction)) or not np.isfinite(float(end_fraction)):
        return np.zeros((0, 2), dtype=np.float64)
    sampled = mqh.resample_polyline_xy(centerline_xy, n_points=max(int(n_points), 31))
    arc = mqh._polyline_arc_length_xy(sampled)
    total = float(arc[-1]) if arc.size else 0.0
    if total <= 0:
        return np.zeros((0, 2), dtype=np.float64)
    fractions = arc / total
    lo, hi = sorted((float(start_fraction), float(end_fraction)))
    keep = (fractions >= lo) & (fractions <= hi)
    seg = sampled[keep]
    if seg.shape[0] < 2:
        seg = np.vstack([
            centerline_point_from_fraction(centerline_xy, lo),
            centerline_point_from_fraction(centerline_xy, hi),
        ])
    return np.asarray(seg, dtype=np.float64)

primary_size_metric_specs = [
    ("positive_area_um2_median", "Median positive area (um^2)"),
]
supplemental_size_metric_specs = [
    ("positive_pixels_median", "Median positive pixels"),
    ("positive_fraction_median", "Median positive fraction"),
]
primary_position_metric_specs = [
    ("posterior_extent_um_median", "Median posterior distance (um)"),
    ("posterior_extent_fraction_median", "Median posterior extent (fraction)"),
]
supplemental_position_metric_specs = [
    ("anterior_extent_fraction_median", "Median anterior extent"),
    ("axis_span_fraction_median", "Median axis span"),
    ("axis_centroid_fraction_median", "Median axis centroid"),
]
primary_bilaterality_metric_specs = [
    ("side_balance_index_median", "Median side-balance index"),
]
supplemental_bilaterality_metric_specs = [
    ("weighted_side_balance_index_median", "Median weighted side-balance index"),
    ("mean_abs_transverse_um_median", "Median abs transverse distance (um)"),
]
pair_order = [("foxf1", "pax8"), ("pax8", "mesp2"), ("foxf1", "mesp2")]

explanation_file_id = representative_ids[min(len(representative_ids) // 2, max(len(representative_ids) - 1, 0))] if representative_ids else None
explanation_row = None
explanation_plane = None
explanation_mask = None
explanation_positive_mask = None
explanation_axis_info = None
explanation_projection = None
explanation_points_xy = np.zeros((0, 2), dtype=np.float64)
explanation_color = None
explanation_marker_name = None
explanation_total_length_px = np.nan
posterior_extent_px = np.nan
anterior_extent_px = np.nan
centroid_arc_px = np.nan
posterior_extent_xy = np.array([np.nan, np.nan], dtype=np.float64)
anterior_extent_xy = np.array([np.nan, np.nan], dtype=np.float64)
centroid_axis_xy = np.array([np.nan, np.nan], dtype=np.float64)
posterior_extent_fraction = np.nan
anterior_extent_fraction = np.nan
centroid_fraction = np.nan
pair_lookup = {}

if explanation_file_id is not None:
    explanation_file_rows = domain_plane_df.loc[
        domain_plane_df["file_id"].astype(int) == int(explanation_file_id)
    ].copy()
    if not explanation_file_rows.empty:
        display_z = int(explanation_file_rows["display_z_index"].iloc[0])
        explanation_plane_rows = explanation_file_rows.loc[
            explanation_file_rows["z_index"].astype(int) == display_z
        ].copy()
        if explanation_plane_rows.empty:
            explanation_plane_rows = explanation_file_rows.loc[
                explanation_file_rows["z_index"].astype(int)
                == int(np.nanmedian(explanation_file_rows["z_index"].astype(float).to_numpy()))
            ].copy()
        if not explanation_plane_rows.empty:
            explanation_plane_rows = explanation_plane_rows.copy()
            explanation_plane_rows["marker_rank"] = (
                explanation_plane_rows["marker_key"].astype(str).map(
                    {"mesp2": 0, "pax8": 1, "foxf1": 2}
                ).fillna(99)
            )
            explanation_plane_rows = explanation_plane_rows.sort_values(
                ["marker_rank", "marker_key"]
            ).reset_index(drop=True)
            explanation_row = explanation_plane_rows.iloc[0]
            explanation_marker_key = str(explanation_row.marker_key)
            explanation_marker_name = str(explanation_row.marker_display_name)
            explanation_color = mqh.MARKER_OUTLINE_COLORS[explanation_marker_key]
            explanation_plane = load_plane_bundle(str(explanation_row.file_path), int(explanation_row.z_index))
            explanation_mask = load_binary_mask_cached(str(explanation_row.mask_path))
            explanation_positive_mask = load_binary_mask_cached(str(explanation_row.positive_mask_path))
            explanation_axis_info = axis_cache[str(explanation_row.file_path)]
            pair_lookup = {
                str(row.marker_key): row
                for row in explanation_plane_rows.itertuples(index=False)
            }

            ys, xs = np.where(explanation_positive_mask)
            explanation_points_xy = (
                np.column_stack([xs, ys]).astype(np.float64)
                if len(xs)
                else np.zeros((0, 2), dtype=np.float64)
            )
            explanation_projection = mqh.project_points_to_centerline(
                explanation_points_xy,
                explanation_axis_info["centerline_xy"],
                n_samples=CENTERLINE_PROJECTION_SAMPLES,
            )
            explanation_total_length_px = float(explanation_projection["total_length_px"])
            posterior_extent_px = (
                float(np.nanmin(explanation_projection["projected_arc_px"]))
                if len(xs)
                else np.nan
            )
            anterior_extent_px = (
                float(np.nanmax(explanation_projection["projected_arc_px"]))
                if len(xs)
                else np.nan
            )
            centroid_arc_px = (
                float(np.nanmean(explanation_projection["projected_arc_px"]))
                if len(xs)
                else np.nan
            )

            posterior_extent_xy = (
                mqh._interpolate_point_along_polyline(
                    explanation_axis_info["centerline_xy"], posterior_extent_px
                )
                if np.isfinite(posterior_extent_px)
                else np.asarray(explanation_axis_info["endpoint_a_xy"], dtype=np.float64)
            )
            anterior_extent_xy = (
                mqh._interpolate_point_along_polyline(
                    explanation_axis_info["centerline_xy"], anterior_extent_px
                )
                if np.isfinite(anterior_extent_px)
                else np.asarray(explanation_axis_info["endpoint_b_xy"], dtype=np.float64)
            )
            centroid_axis_xy = (
                mqh._interpolate_point_along_polyline(
                    explanation_axis_info["centerline_xy"], centroid_arc_px
                )
                if np.isfinite(centroid_arc_px)
                else np.asarray(explanation_axis_info["midpoint_xy"], dtype=np.float64)
            )

            posterior_extent_fraction = (
                posterior_extent_px / explanation_total_length_px
                if explanation_total_length_px > 0 and np.isfinite(posterior_extent_px)
                else np.nan
            )
            anterior_extent_fraction = (
                anterior_extent_px / explanation_total_length_px
                if explanation_total_length_px > 0 and np.isfinite(anterior_extent_px)
                else np.nan
            )
            centroid_fraction = (
                centroid_arc_px / explanation_total_length_px
                if explanation_total_length_px > 0 and np.isfinite(centroid_arc_px)
                else np.nan
            )

display(Markdown("### Primary Figures Aligned To The Original Target List"))

if explanation_row is not None:
    positive_fraction = float(explanation_row.positive_fraction)
    positive_pixels = int(explanation_row.positive_pixels)
    organoid_pixels = int(explanation_row.organoid_pixel_count)
    positive_area_um2 = float(explanation_row.positive_area_um2)

    fig, axes = plt.subplots(1, 2, figsize=(14.6, 4.8), constrained_layout=True)
    ax_img, ax_diag = axes

    ax_img.imshow(mqh.robust_rescale(explanation_plane["dapi"]), cmap="gray", interpolation="nearest")
    mqh.plot_mask_outline(ax_img, explanation_mask, color="white", linewidth=0.9)
    mqh.plot_mask_outline(ax_img, explanation_positive_mask, color=explanation_color, linewidth=1.5)
    ax_img.text(14, 18, "positive region", color=explanation_color, fontsize=8.6, weight="bold")
    ax_img.set_title(
        f"Representative image | file {int(explanation_file_id):02d} | z={int(explanation_row.z_index):02d} | {explanation_marker_name}",
        fontsize=10.5,
    )
    ax_img.set_xticks([])
    ax_img.set_yticks([])

    configure_geometry_panel(ax_diag, explanation_mask)
    draw_mask_fill(ax_diag, explanation_mask, "0.86", alpha=0.85)
    draw_mask_fill(ax_diag, explanation_positive_mask, explanation_color, alpha=0.55)
    mqh.plot_mask_outline(ax_diag, explanation_mask, color="0.35", linewidth=1.0)
    mqh.plot_mask_outline(ax_diag, explanation_positive_mask, color=explanation_color, linewidth=1.6)
    mask_ys, mask_xs = np.where(explanation_mask)
    mask_centroid_xy = np.array(
        [float(np.nanmean(mask_xs)), float(np.nanmean(mask_ys))],
        dtype=np.float64,
    )
    positive_centroid_xy = np.array(
        [float(explanation_row.centroid_x_px), float(explanation_row.centroid_y_px)],
        dtype=np.float64,
    )
    ax_diag.annotate(
        "organoid mask area",
        xy=(mask_centroid_xy[0], mask_centroid_xy[1]),
        xytext=(0.12, 0.80),
        textcoords="axes fraction",
        fontsize=8.8,
        color="0.25",
        ha="left",
        va="center",
        bbox=dict(boxstyle="round,pad=0.20", facecolor="white", edgecolor="0.85", alpha=0.96),
        arrowprops=dict(arrowstyle="-", color="0.35", linewidth=1.0),
    )
    ax_diag.annotate(
        "positive region",
        xy=(positive_centroid_xy[0], positive_centroid_xy[1]),
        xytext=(0.76, 0.52),
        textcoords="axes fraction",
        fontsize=8.8,
        color=explanation_color,
        ha="left",
        va="center",
        bbox=dict(boxstyle="round,pad=0.20", facecolor="white", edgecolor="0.85", alpha=0.96),
        arrowprops=dict(arrowstyle="-", color=explanation_color, linewidth=1.0),
    )
    ax_diag.text(
        0.03,
        0.04,
        f"positive pixels = {positive_pixels}\npositive fraction = {positive_pixels} / {organoid_pixels} = {positive_fraction:.3f}\npositive area = {positive_area_um2:.1f} um^2",
        transform=ax_diag.transAxes,
        fontsize=9.1,
        va="bottom",
        ha="left",
        color="0.1",
        bbox=dict(boxstyle="round,pad=0.28", facecolor="white", edgecolor="0.85", alpha=0.95),
    )
    ax_diag.set_title("What the size metrics measure", fontsize=10.5)

    fig.savefig(DOMAIN_SIZE_EXPLAINER_PATH, dpi=180, bbox_inches="tight")
    if not OPTIMIZATION_MODE:
        display(fig)
    plt.close(fig)
    print("Wrote size explainer figure:", DOMAIN_SIZE_EXPLAINER_PATH.relative_to(ROOT).as_posix())

fig, axes = plt.subplots(1, len(primary_size_metric_specs), figsize=(5.4, 4.2), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, (metric_name, title) in zip(axes, primary_size_metric_specs):
    plot_marker_metric_panel(ax, metric_name, title)
fig.savefig(DOMAIN_SIZE_SUMMARY_PATH, dpi=180, bbox_inches="tight")
if not OPTIMIZATION_MODE:
    display(fig)
plt.close(fig)

if explanation_row is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13.8, 4.8), constrained_layout=True)
    ax_img, ax_diag = axes

    ax_img.imshow(mqh.robust_rescale(explanation_plane["dapi"]), cmap="gray", interpolation="nearest")
    mqh.plot_mask_outline(ax_img, explanation_mask, color="white", linewidth=0.9)
    mqh.plot_mask_outline(ax_img, explanation_positive_mask, color=explanation_color, linewidth=1.5)
    mqh.plot_centerline_overlay(
        ax=ax_img,
        centerline_xy=explanation_axis_info["centerline_xy"],
        midpoint_xy=explanation_axis_info["midpoint_xy"],
        endpoint_a_xy=explanation_axis_info["endpoint_a_xy"],
        endpoint_b_xy=explanation_axis_info["endpoint_b_xy"],
        base_endpoint_a_xy=explanation_axis_info["base_endpoint_a_xy"],
        base_endpoint_b_xy=explanation_axis_info["base_endpoint_b_xy"],
        posterior_click_xy=np.asarray(explanation_axis_info["posterior_click_xy"], dtype=np.float64),
        line_width=1.2,
    )
    ax_img.scatter([posterior_extent_xy[0]], [posterior_extent_xy[1]], c=[explanation_color], s=34, marker="s", edgecolors="white", linewidths=0.5)
    ax_img.scatter([anterior_extent_xy[0]], [anterior_extent_xy[1]], c=[explanation_color], s=34, marker="^", edgecolors="white", linewidths=0.5)
    ax_img.scatter([centroid_axis_xy[0]], [centroid_axis_xy[1]], c=[explanation_color], s=40, marker="o", edgecolors="white", linewidths=0.5)
    ax_img.text(posterior_extent_xy[0] + 10, posterior_extent_xy[1] - 10, "posterior extent", color=explanation_color, fontsize=8.0, weight="bold")
    ax_img.text(anterior_extent_xy[0] + 10, anterior_extent_xy[1] - 10, "anterior extent", color=explanation_color, fontsize=8.0, weight="bold")
    ax_img.text(centroid_axis_xy[0] + 10, centroid_axis_xy[1] + 14, "centroid", color=explanation_color, fontsize=8.0, weight="bold")
    ax_img.set_title(
        f"Representative image | file {int(explanation_file_id):02d} | z={int(explanation_row.z_index):02d} | {explanation_marker_name}",
        fontsize=10.5,
    )
    ax_img.set_xticks([])
    ax_img.set_yticks([])

    configure_geometry_panel(ax_diag, explanation_mask)
    draw_mask_fill(ax_diag, explanation_mask, "0.96", alpha=0.92)
    mqh.plot_mask_outline(ax_diag, explanation_mask, color="0.75", linewidth=0.9)
    ax_diag.plot(
        explanation_axis_info["centerline_xy"][:, 0],
        explanation_axis_info["centerline_xy"][:, 1],
        color="black",
        linewidth=2.0,
        alpha=0.85,
    )
    if np.isfinite(posterior_extent_fraction) and np.isfinite(anterior_extent_fraction):
        interval_xy = centerline_segment_xy(
            explanation_axis_info["centerline_xy"],
            posterior_extent_fraction,
            anterior_extent_fraction,
        )
        if interval_xy.shape[0] >= 2:
            ax_diag.plot(
                interval_xy[:, 0],
                interval_xy[:, 1],
                color=explanation_color,
                linewidth=6.0,
                solid_capstyle="round",
                alpha=0.88,
            )
        ax_diag.scatter(
            [posterior_extent_xy[0]],
            [posterior_extent_xy[1]],
            color=explanation_color,
            s=48,
            marker="s",
            edgecolors="white",
            linewidths=0.6,
            zorder=4,
        )
        ax_diag.scatter(
            [anterior_extent_xy[0]],
            [anterior_extent_xy[1]],
            color=explanation_color,
            s=48,
            marker="^",
            edgecolors="white",
            linewidths=0.6,
            zorder=4,
        )
    if np.isfinite(centroid_fraction):
        ax_diag.scatter(
            [centroid_axis_xy[0]],
            [centroid_axis_xy[1]],
            color=explanation_color,
            s=58,
            marker="o",
            edgecolors="white",
            linewidths=0.6,
            zorder=5,
        )
    ax_diag.scatter(
        [explanation_axis_info["endpoint_a_xy"][0], explanation_axis_info["endpoint_b_xy"][0]],
        [explanation_axis_info["endpoint_a_xy"][1], explanation_axis_info["endpoint_b_xy"][1]],
        color="black",
        s=34,
        zorder=4,
    )
    ax_diag.annotate(
        "posterior end",
        xy=tuple(explanation_axis_info["endpoint_a_xy"]),
        xytext=(0.08, 0.15),
        textcoords="axes fraction",
        fontsize=8.6,
        color="0.15",
        ha="left",
        va="center",
        bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor="0.85", alpha=0.96),
        arrowprops=dict(arrowstyle="-", color="0.25", linewidth=1.0),
    )
    ax_diag.annotate(
        "anterior end",
        xy=tuple(explanation_axis_info["endpoint_b_xy"]),
        xytext=(0.60, 0.80),
        textcoords="axes fraction",
        fontsize=8.6,
        color="0.15",
        ha="left",
        va="center",
        bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor="0.85", alpha=0.96),
        arrowprops=dict(arrowstyle="-", color="0.25", linewidth=1.0),
    )
    ax_diag.annotate(
        "posterior extent fraction",
        xy=tuple(posterior_extent_xy),
        xytext=(0.05, 0.32),
        textcoords="axes fraction",
        fontsize=8.4,
        color=explanation_color,
        ha="left",
        va="center",
        bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor="0.85", alpha=0.96),
        arrowprops=dict(arrowstyle="-", color=explanation_color, linewidth=1.0),
    )
    ax_diag.annotate(
        "anterior extent fraction",
        xy=tuple(anterior_extent_xy),
        xytext=(0.60, 0.86),
        textcoords="axes fraction",
        fontsize=8.4,
        color=explanation_color,
        ha="left",
        va="center",
        bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor="0.85", alpha=0.96),
        arrowprops=dict(arrowstyle="-", color=explanation_color, linewidth=1.0),
    )
    ax_diag.annotate(
        "centroid fraction",
        xy=tuple(centroid_axis_xy),
        xytext=(0.63, 0.56),
        textcoords="axes fraction",
        fontsize=8.6,
        color=explanation_color,
        ha="left",
        va="center",
        bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor="0.85", alpha=0.96),
        arrowprops=dict(arrowstyle="-", color=explanation_color, linewidth=1.0),
    )
    ax_diag.text(
        0.58,
        0.63,
        "axis span fraction",
        transform=ax_diag.transAxes,
        color=explanation_color,
        fontsize=8.8,
        weight="bold",
        ha="center",
        va="center",
        bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor="0.85", alpha=0.96),
    )
    ax_diag.set_title("What the axis-position metrics measure", fontsize=10.5)

    fig.savefig(DOMAIN_POSITION_EXPLAINER_PATH, dpi=180, bbox_inches="tight")
    if not OPTIMIZATION_MODE:
        display(fig)
    plt.close(fig)
    print("Wrote axis-position explainer figure:", DOMAIN_POSITION_EXPLAINER_PATH.relative_to(ROOT).as_posix())

fig, axes = plt.subplots(1, len(primary_position_metric_specs), figsize=(10.4, 4.2), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, (metric_name, title) in zip(axes, primary_position_metric_specs):
    plot_marker_metric_panel(ax, metric_name, title)
fig.savefig(DOMAIN_POSITION_SUMMARY_PATH, dpi=180, bbox_inches="tight")
if not OPTIMIZATION_MODE:
    display(fig)
plt.close(fig)

if explanation_row is not None and explanation_points_xy.shape[0] > 0:
    sample_count = min(48, int(explanation_points_xy.shape[0]))
    sample_idx = np.unique(
        np.linspace(0, explanation_points_xy.shape[0] - 1, sample_count).astype(int)
    )
    sample_points_xy = explanation_points_xy[sample_idx]
    sample_projection = mqh.project_points_to_centerline(
        sample_points_xy,
        explanation_axis_info["centerline_xy"],
        n_samples=CENTERLINE_PROJECTION_SAMPLES,
    )
    sample_projected_xy = np.asarray(sample_projection["projected_xy"], dtype=np.float64)
    sample_signed_px = np.asarray(sample_projection["signed_transverse_px"], dtype=np.float64)
    sample_colors = np.where(
        sample_signed_px < -0.5,
        "#3b82f6",
        np.where(sample_signed_px > 0.5, "#f97316", "#a3a3a3"),
    )

    fig, axes = plt.subplots(1, 2, figsize=(13.8, 4.8), constrained_layout=True)
    ax_img, ax_diag = axes

    ax_img.imshow(mqh.robust_rescale(explanation_plane["dapi"]), cmap="gray", interpolation="nearest")
    mqh.plot_mask_outline(ax_img, explanation_mask, color="white", linewidth=0.9)
    mqh.plot_mask_outline(ax_img, explanation_positive_mask, color=explanation_color, linewidth=1.2)
    mqh.plot_centerline_overlay(
        ax=ax_img,
        centerline_xy=explanation_axis_info["centerline_xy"],
        midpoint_xy=explanation_axis_info["midpoint_xy"],
        endpoint_a_xy=explanation_axis_info["endpoint_a_xy"],
        endpoint_b_xy=explanation_axis_info["endpoint_b_xy"],
        base_endpoint_a_xy=explanation_axis_info["base_endpoint_a_xy"],
        base_endpoint_b_xy=explanation_axis_info["base_endpoint_b_xy"],
        posterior_click_xy=np.asarray(explanation_axis_info["posterior_click_xy"], dtype=np.float64),
        line_width=1.2,
    )
    for pt_xy, proj_xy, color in zip(sample_points_xy, sample_projected_xy, sample_colors):
        ax_img.plot(
            [pt_xy[0], proj_xy[0]],
            [pt_xy[1], proj_xy[1]],
            color=color,
            alpha=0.28,
            linewidth=0.8,
        )
    ax_img.scatter(
        sample_points_xy[:, 0],
        sample_points_xy[:, 1],
        c=sample_colors.tolist(),
        s=10,
        alpha=0.85,
        edgecolors="none",
    )
    ax_img.set_title(
        f"Representative image | file {int(explanation_file_id):02d} | z={int(explanation_row.z_index):02d} | {explanation_marker_name}",
        fontsize=10.5,
    )
    ax_img.set_xticks([])
    ax_img.set_yticks([])

    configure_geometry_panel(ax_diag, explanation_mask)
    draw_mask_fill(ax_diag, explanation_mask, "0.97", alpha=0.94)
    mqh.plot_mask_outline(ax_diag, explanation_mask, color="0.82", linewidth=0.9)
    mqh.plot_mask_outline(ax_diag, explanation_positive_mask, color=explanation_color, linewidth=1.2)
    ax_diag.plot(
        explanation_axis_info["centerline_xy"][:, 0],
        explanation_axis_info["centerline_xy"][:, 1],
        color="black",
        linewidth=1.8,
        alpha=0.85,
    )
    for pt_xy, proj_xy, color in zip(sample_points_xy, sample_projected_xy, sample_colors):
        ax_diag.plot(
            [pt_xy[0], proj_xy[0]],
            [pt_xy[1], proj_xy[1]],
            color=color,
            alpha=0.30,
            linewidth=0.9,
        )
    ax_diag.scatter(
        sample_points_xy[:, 0],
        sample_points_xy[:, 1],
        c=sample_colors.tolist(),
        s=13,
        alpha=0.92,
        edgecolors="white",
        linewidths=0.2,
        zorder=4,
    )
    if sample_projected_xy.shape[0] > 0:
        median_idx = int(np.argmin(np.abs(np.asarray(sample_projection["transverse_abs_px"]) - np.nanmedian(np.asarray(sample_projection["transverse_abs_px"])))))
        ax_diag.plot(
            [sample_points_xy[median_idx, 0], sample_projected_xy[median_idx, 0]],
            [sample_points_xy[median_idx, 1], sample_projected_xy[median_idx, 1]],
            color="0.25",
            linewidth=1.3,
        )
        mid_pt = 0.5 * (sample_points_xy[median_idx] + sample_projected_xy[median_idx])
        ax_diag.annotate(
            "abs transverse distance",
            xy=tuple(mid_pt),
            xytext=(0.67, 0.58),
            textcoords="axes fraction",
            fontsize=8.6,
            color="0.2",
            ha="left",
            va="center",
            bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor="0.85", alpha=0.96),
            arrowprops=dict(arrowstyle="-", color="0.25", linewidth=1.0),
        )
    ax_diag.text(
        0.03,
        0.04,
        (
            f"negative-side pixels = {int(explanation_row.left_positive_pixels)}\n"
            f"positive-side pixels = {int(explanation_row.right_positive_pixels)}\n"
            f"negative-side weighted signal = {float(explanation_row.left_positive_weighted_signal):.1f}\n"
            f"positive-side weighted signal = {float(explanation_row.right_positive_weighted_signal):.1f}\n"
            f"mean abs transverse distance = {float(explanation_row.mean_abs_transverse_um):.1f} um\n"
            f"side balance index = {float(explanation_row.side_balance_index):.3f}\n"
            f"weighted side balance index = {float(explanation_row.weighted_side_balance_index):.3f}"
        ),
        transform=ax_diag.transAxes,
        fontsize=9.0,
        va="bottom",
        ha="left",
        color="0.1",
        bbox=dict(boxstyle="round,pad=0.28", facecolor="white", edgecolor="0.85", alpha=0.95),
    )
    ax_diag.text(0.03, 0.92, "negative side", transform=ax_diag.transAxes, color="#3b82f6", fontsize=8.8, weight="bold", ha="left")
    ax_diag.text(0.97, 0.92, "positive side", transform=ax_diag.transAxes, color="#f97316", fontsize=8.8, weight="bold", ha="right")
    ax_diag.set_title("What the transverse / bilaterality metrics measure", fontsize=10.5)

    fig.savefig(DOMAIN_BILATERALITY_EXPLAINER_PATH, dpi=180, bbox_inches="tight")
    if not OPTIMIZATION_MODE:
        display(fig)
    plt.close(fig)
    print("Wrote bilaterality explainer figure:", DOMAIN_BILATERALITY_EXPLAINER_PATH.relative_to(ROOT).as_posix())

display(
    Markdown(
        '''
        #### How bilaterality is computed

        For one marker on one z plane:

        - project each positive pixel onto the posterior-oriented axis and assign it a signed transverse offset
        - count `left_count = signed_offset < -0.5 px`
        - count `right_count = signed_offset > 0.5 px`
        - pixels very close to the axis are not included in the left/right denominator
        - the original count-based metric is:

        `side_balance_index = 1 - abs(right_count - left_count) / (right_count + left_count)`

        - the weighted metric uses only positive pixels and weights each positive pixel by:

        `w_i = max(corrected_intensity_i - threshold_value, 0)`

        - then computes:

        `weighted_side_balance_index = 1 - abs(W_right - W_left) / (W_right + W_left)`

        where `W_left` and `W_right` are the left/right sums of `w_i`

        Interpretation:

        - for both metrics, `1.0` = perfectly balanced left/right signal
        - for both metrics, `0.0` = completely one-sided signal
        - swapping which side is called positive vs negative does **not** change either metric

        Both are different from `mean abs transverse distance`, which measures how far the signal sits from the axis rather than how evenly it is split across the two sides.
        '''
    )
)

fig, axes = plt.subplots(1, len(primary_bilaterality_metric_specs), figsize=(5.4, 4.2), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, (metric_name, title) in zip(axes, primary_bilaterality_metric_specs):
    plot_marker_metric_panel(ax, metric_name, title)
fig.savefig(DOMAIN_BILATERALITY_SUMMARY_PATH, dpi=180, bbox_inches="tight")
if not OPTIMIZATION_MODE:
    display(fig)
plt.close(fig)

display(Markdown("### Supplementary Descriptive Figures"))

fig, axes = plt.subplots(1, len(supplemental_size_metric_specs), figsize=(10.0, 4.2), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, (metric_name, title) in zip(axes, supplemental_size_metric_specs):
    plot_marker_metric_panel(ax, metric_name, title)
fig.savefig(DOMAIN_SIZE_SUPPLEMENT_PATH, dpi=180, bbox_inches="tight")
if not OPTIMIZATION_MODE:
    display(fig)
plt.close(fig)

fig, axes = plt.subplots(1, len(supplemental_position_metric_specs), figsize=(14.2, 4.2), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, (metric_name, title) in zip(axes, supplemental_position_metric_specs):
    plot_marker_metric_panel(ax, metric_name, title)
fig.savefig(DOMAIN_POSITION_SUPPLEMENT_PATH, dpi=180, bbox_inches="tight")
if not OPTIMIZATION_MODE:
    display(fig)
plt.close(fig)

fig, axes = plt.subplots(1, len(supplemental_bilaterality_metric_specs), figsize=(5.8, 4.2), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, (metric_name, title) in zip(axes, supplemental_bilaterality_metric_specs):
    plot_marker_metric_panel(ax, metric_name, title)
fig.savefig(DOMAIN_BILATERALITY_SUPPLEMENT_PATH, dpi=180, bbox_inches="tight")
if not OPTIMIZATION_MODE:
    display(fig)
plt.close(fig)

display(Markdown("#### Pairwise Marker Relationships"))

explanation_pair = None
if explanation_row is not None:
    for pair in pair_order:
        if pair[0] in pair_lookup and pair[1] in pair_lookup:
            explanation_pair = pair
            break

if explanation_pair is not None:
    row_a = pair_lookup[explanation_pair[0]]
    row_b = pair_lookup[explanation_pair[1]]
    color_a = mqh.MARKER_OUTLINE_COLORS[explanation_pair[0]]
    color_b = mqh.MARKER_OUTLINE_COLORS[explanation_pair[1]]
    mask_a = load_binary_mask_cached(str(row_a.positive_mask_path))
    mask_b = load_binary_mask_cached(str(row_b.positive_mask_path))
    overlap_start = max(float(row_a.posterior_extent_fraction), float(row_b.posterior_extent_fraction))
    overlap_end = min(float(row_a.anterior_extent_fraction), float(row_b.anterior_extent_fraction))
    overlap_fraction = max(0.0, overlap_end - overlap_start)

    fig, axes = plt.subplots(1, 2, figsize=(13.8, 4.8), constrained_layout=True)
    ax_img, ax_diag = axes

    ax_img.imshow(mqh.robust_rescale(explanation_plane["dapi"]), cmap="gray", interpolation="nearest")
    mqh.plot_mask_outline(ax_img, explanation_mask, color="white", linewidth=0.9)
    mqh.plot_centerline_overlay(
        ax=ax_img,
        centerline_xy=explanation_axis_info["centerline_xy"],
        midpoint_xy=explanation_axis_info["midpoint_xy"],
        endpoint_a_xy=explanation_axis_info["endpoint_a_xy"],
        endpoint_b_xy=explanation_axis_info["endpoint_b_xy"],
        base_endpoint_a_xy=explanation_axis_info["base_endpoint_a_xy"],
        base_endpoint_b_xy=explanation_axis_info["base_endpoint_b_xy"],
        posterior_click_xy=np.asarray(explanation_axis_info["posterior_click_xy"], dtype=np.float64),
        line_width=1.2,
    )
    mqh.plot_mask_outline(ax_img, mask_a, color=color_a, linewidth=1.4)
    mqh.plot_mask_outline(ax_img, mask_b, color=color_b, linewidth=1.4)
    ax_img.set_title(
        f"Representative image | file {int(explanation_file_id):02d} | z={int(explanation_row.z_index):02d} | {mqh.marker_display_name(explanation_pair[0])} vs {mqh.marker_display_name(explanation_pair[1])}",
        fontsize=10.0,
    )
    ax_img.set_xticks([])
    ax_img.set_yticks([])

    ax_diag.set_xlim(0.0, 1.0)
    ax_diag.set_ylim(-0.05, 1.05)
    ax_diag.axis("off")
    axis_x = 0.50
    strip_a_x = 0.36
    strip_b_x = 0.64
    overlap_x0 = 0.44
    overlap_x1 = 0.56
    posterior_a = float(row_a.posterior_extent_fraction)
    anterior_a = float(row_a.anterior_extent_fraction)
    posterior_b = float(row_b.posterior_extent_fraction)
    anterior_b = float(row_b.anterior_extent_fraction)
    centroid_a = float(row_a.axis_centroid_fraction)
    centroid_b = float(row_b.axis_centroid_fraction)

    ax_diag.plot([axis_x, axis_x], [0.0, 1.0], color="0.20", linewidth=1.8, zorder=1)
    ax_diag.scatter([axis_x, axis_x], [0.0, 1.0], s=32, color="0.20", zorder=2)
    ax_diag.text(axis_x, -0.03, "posterior = 0", ha="center", va="top", fontsize=9.2, color="0.20")
    ax_diag.text(axis_x, 1.03, "anterior = 1", ha="center", va="bottom", fontsize=9.2, color="0.20")

    if overlap_fraction > 0:
        ax_diag.fill_betweenx(
            [overlap_start, overlap_end],
            overlap_x0,
            overlap_x1,
            color="0.72",
            alpha=0.65,
            zorder=1,
        )
        ax_diag.text(
            0.50,
            0.5 * (overlap_start + overlap_end),
            "overlap",
            ha="center",
            va="center",
            fontsize=8.5,
            color="0.22",
            bbox=dict(boxstyle="round,pad=0.14", facecolor="white", edgecolor="0.88", alpha=0.95),
        )

    ax_diag.plot([strip_a_x, strip_a_x], [posterior_a, anterior_a], color=color_a, linewidth=6.5, solid_capstyle="round", zorder=3)
    ax_diag.plot([strip_b_x, strip_b_x], [posterior_b, anterior_b], color=color_b, linewidth=6.5, solid_capstyle="round", zorder=3)
    ax_diag.scatter([strip_a_x, strip_a_x], [posterior_a, anterior_a], s=28, color=color_a, edgecolors="white", linewidths=0.5, zorder=4)
    ax_diag.scatter([strip_b_x, strip_b_x], [posterior_b, anterior_b], s=28, color=color_b, edgecolors="white", linewidths=0.5, zorder=4)
    ax_diag.scatter([strip_a_x], [centroid_a], s=78, color=color_a, edgecolors="white", linewidths=0.7, zorder=5)
    ax_diag.scatter([strip_b_x], [centroid_b], s=78, color=color_b, edgecolors="white", linewidths=0.7, zorder=5)
    ax_diag.plot([strip_a_x, strip_b_x], [centroid_a, centroid_b], color="0.35", linewidth=1.1, linestyle="--", zorder=2)
    ax_diag.plot([strip_a_x, strip_b_x], [posterior_a, posterior_b], color="0.45", linewidth=1.0, linestyle=":", zorder=2)
    ax_diag.plot([strip_a_x, strip_b_x], [anterior_a, anterior_b], color="0.45", linewidth=1.0, linestyle=":", zorder=2)

    ax_diag.annotate(
        f"{mqh.marker_display_name(explanation_pair[0])}\ninterval",
        xy=(strip_a_x, centroid_a),
        xytext=(0.05, min(0.90, centroid_a + 0.10)),
        fontsize=9.0,
        color=color_a,
        ha="left",
        va="center",
        bbox=dict(boxstyle="round,pad=0.20", facecolor="white", edgecolor="0.85", alpha=0.97),
        arrowprops=dict(arrowstyle="-", color=color_a, linewidth=1.0),
    )
    ax_diag.annotate(
        f"{mqh.marker_display_name(explanation_pair[1])}\ninterval",
        xy=(strip_b_x, centroid_b),
        xytext=(0.79, min(0.92, centroid_b + 0.10)),
        fontsize=9.0,
        color=color_b,
        ha="left",
        va="center",
        bbox=dict(boxstyle="round,pad=0.20", facecolor="white", edgecolor="0.85", alpha=0.97),
        arrowprops=dict(arrowstyle="-", color=color_b, linewidth=1.0),
    )
    ax_diag.annotate(
        "centroid delta\n(B - A)",
        xy=(0.50, 0.5 * (centroid_a + centroid_b)),
        xytext=(0.79, 0.53),
        fontsize=8.8,
        color="0.20",
        ha="left",
        va="center",
        bbox=dict(boxstyle="round,pad=0.20", facecolor="white", edgecolor="0.85", alpha=0.97),
        arrowprops=dict(arrowstyle="-", color="0.35", linewidth=1.0),
    )
    ax_diag.annotate(
        "anterior delta\n(B - A)",
        xy=(0.50, 0.5 * (anterior_a + anterior_b)),
        xytext=(0.05, min(0.96, 0.5 * (anterior_a + anterior_b) + 0.06)),
        fontsize=8.6,
        color="0.20",
        ha="left",
        va="center",
        bbox=dict(boxstyle="round,pad=0.20", facecolor="white", edgecolor="0.85", alpha=0.97),
        arrowprops=dict(arrowstyle="-", color="0.35", linewidth=1.0),
    )
    ax_diag.annotate(
        "posterior delta\n(B - A)",
        xy=(0.50, 0.5 * (posterior_a + posterior_b)),
        xytext=(0.79, max(0.12, 0.5 * (posterior_a + posterior_b) - 0.05)),
        fontsize=8.6,
        color="0.20",
        ha="left",
        va="center",
        bbox=dict(boxstyle="round,pad=0.20", facecolor="white", edgecolor="0.85", alpha=0.97),
        arrowprops=dict(arrowstyle="-", color="0.35", linewidth=1.0),
    )
    ax_diag.text(
        0.05,
        0.06,
        f"centroid delta = {float(row_b.axis_centroid_fraction) - float(row_a.axis_centroid_fraction):+.3f}\nposterior delta = {float(row_b.posterior_extent_fraction) - float(row_a.posterior_extent_fraction):+.3f}\nanterior delta = {float(row_b.anterior_extent_fraction) - float(row_a.anterior_extent_fraction):+.3f}\ninterval overlap = {overlap_fraction:.3f}",
        transform=ax_diag.transAxes,
        fontsize=8.9,
        va="bottom",
        ha="left",
        color="0.1",
        bbox=dict(boxstyle="round,pad=0.28", facecolor="white", edgecolor="0.85", alpha=0.95),
    )
    ax_diag.set_title("What the pairwise metrics actually compare", fontsize=10.5)

    fig.savefig(PAIRWISE_EXPLAINER_PATH, dpi=180, bbox_inches="tight")
    if not OPTIMIZATION_MODE:
        display(fig)
    plt.close(fig)
    print("Wrote pairwise explainer figure:", PAIRWISE_EXPLAINER_PATH.relative_to(ROOT).as_posix())

fig, axes = plt.subplots(4, 3, figsize=(14.0, 13.0), constrained_layout=True)
pair_metric_specs = [
    ("centroid_delta_fraction_b_minus_a_median", "Centroid delta"),
    ("posterior_extent_delta_fraction_b_minus_a_median", "Posterior extent delta"),
    ("anterior_extent_delta_fraction_b_minus_a_median", "Anterior extent delta"),
    ("axis_interval_overlap_fraction_median", "Interval overlap"),
]
for row_idx, (metric_name, row_label) in enumerate(pair_metric_specs):
    for col_idx, pair in enumerate(pair_order):
        ax = axes[row_idx, col_idx]
        sub = pairwise_file_df.loc[
            (pairwise_file_df["marker_a_key"] == pair[0])
            & (pairwise_file_df["marker_b_key"] == pair[1])
        ].copy()
        if sub.empty:
            ax.axis("off")
            continue
        y = sub[metric_name].astype(float).to_numpy()
        x = np.arange(len(y), dtype=float)
        zero_centered = "delta" in metric_name
        if zero_centered:
            ax.axhline(0.0, color="0.4", linestyle="--", linewidth=1.0)
        ax.scatter(x, y, color="tab:purple", s=24, alpha=0.85, edgecolors="white", linewidths=0.3)
        if row_idx == 0:
            ax.set_title(
                f"{mqh.marker_display_name(pair[1])} - {mqh.marker_display_name(pair[0])}",
                fontsize=10.0,
            )
        if col_idx == 0:
            ax.set_ylabel(row_label)
        if row_idx == len(pair_metric_specs) - 1:
            ax.set_xlabel("File rank")
        ax.grid(alpha=0.18)

fig.savefig(PAIRWISE_SUMMARY_PATH, dpi=180, bbox_inches="tight")
if not OPTIMIZATION_MODE:
    display(fig)
plt.close(fig)

print("Wrote domain-size summary figure:", DOMAIN_SIZE_SUMMARY_PATH.relative_to(ROOT).as_posix())
print("Wrote axis-position summary figure:", DOMAIN_POSITION_SUMMARY_PATH.relative_to(ROOT).as_posix())
print("Wrote bilaterality summary figure:", DOMAIN_BILATERALITY_SUMMARY_PATH.relative_to(ROOT).as_posix())
print("Wrote size supplement figure:", DOMAIN_SIZE_SUPPLEMENT_PATH.relative_to(ROOT).as_posix())
print("Wrote axis-position supplement figure:", DOMAIN_POSITION_SUPPLEMENT_PATH.relative_to(ROOT).as_posix())
print("Wrote bilaterality supplement figure:", DOMAIN_BILATERALITY_SUPPLEMENT_PATH.relative_to(ROOT).as_posix())
print("Wrote pairwise summary figure:", PAIRWISE_SUMMARY_PATH.relative_to(ROOT).as_posix())


## Save Stage Outputs


In [ ]:
created_paths = [
    AXIS_TABLE_PATH,
    DOMAIN_PLANE_TABLE_PATH,
    DOMAIN_FILE_TABLE_PATH,
    PAIRWISE_PLANE_TABLE_PATH,
    PAIRWISE_FILE_TABLE_PATH,
    DOMAIN_OVERLAY_PATH,
    DOMAIN_SIZE_EXPLAINER_PATH,
    DOMAIN_SIZE_SUMMARY_PATH,
    DOMAIN_SIZE_SUPPLEMENT_PATH,
    DOMAIN_POSITION_EXPLAINER_PATH,
    DOMAIN_POSITION_SUMMARY_PATH,
    DOMAIN_POSITION_SUPPLEMENT_PATH,
    DOMAIN_BILATERALITY_EXPLAINER_PATH,
    DOMAIN_BILATERALITY_SUMMARY_PATH,
    DOMAIN_BILATERALITY_SUPPLEMENT_PATH,
    PAIRWISE_EXPLAINER_PATH,
    PAIRWISE_SUMMARY_PATH,
]

print("Created files")
for path in created_paths:
    if path.exists():
        print("-", path.relative_to(ROOT).as_posix())


## Next Step

Use these per-file domain summaries as the input to the final stats / figure-export notebook, where sample sizes, plots, and statistical tests can be reported explicitly.
